# Practical Application 3 — Comparing Classifiers
## Assignment 17.1 | UCI Bank Marketing

**Business question.** Which customers should the Portuguese bank prioritize for a term-deposit telemarketing campaign, using only information available before a call is placed?

**Target:** `y` (`yes` = subscribed). The positive class is the minority class, so accuracy alone is not a useful selection criterion.

**Method:** CRISP-DM framing, exploratory analysis, leakage-aware preprocessing, stratified holdout, four classifiers (KNN, Logistic Regression, Decision Tree, SVM), 5-fold cross-validation and hyperparameter search, and interpretation.

> Important design decision: `duration` is excluded from the primary models. It is only known after a call has occurred and is therefore unavailable at the intended pre-call decision point. Results from a model using it would not represent a deployable prospect-prioritization model.

## 1. Setup and reproducibility

Run from the repository root. The included CSV is the UCI `bank-additional-full.csv` dataset (41,188 records; 20 predictors plus target).

In [ ]:
import warnings, time, sys
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

## 2. Business understanding

The bank must decide which prospects merit scarce outbound calling capacity. A false positive consumes agent time on a customer who does not subscribe; a false negative misses a potential deposit and its lifetime value.

The operational threshold should therefore reflect the relative cost of calls, available agent capacity, and value of a subscription. This assignment uses **ROC-AUC** as the primary tuning metric because it measures ranking quality across thresholds. **Average precision (PR-AUC)**, positive-class recall, precision, and F1 are also reported because only about one in nine customers subscribes. Accuracy is included for context, not as the optimization objective.

A naïve classifier that always predicts `no` obtains the majority-class accuracy (about 88.7%) while having zero recall for subscribers.

## 3. Load and validate data

In [ ]:
import os, shutil, urllib.request, zipfile
from pathlib import Path

DATA_PATH = Path("data/bank-additional-full.csv")
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    archive_path = DATA_PATH.parent / "bank-marketing.zip"
    source_url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
    print("Dataset not found locally. Downloading the UCI Bank Marketing archive...")
    urllib.request.urlretrieve(source_url, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        csv_member = next(name for name in archive.namelist() if name.endswith("bank-additional-full.csv"))
        with archive.open(csv_member) as source, DATA_PATH.open("wb") as target:
            shutil.copyfileobj(source, target)
    archive_path.unlink(missing_ok=True)

df = pd.read_csv(DATA_PATH, sep=";")

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())
display(df.dtypes.rename("dtype").to_frame())
display(df["y"].value_counts().rename_axis("subscription").to_frame("count"))
display((df["y"].value_counts(normalize=True) * 100).round(2).rename("percent").to_frame())

### Data quality and unknown categories

The data dictionary identifies `unknown` as a valid recorded category for several categorical fields. These are not silently treated as missing numeric values. The primary workflow preserves `unknown` as an explicit category, which retains information about missing/undisclosed responses and avoids deleting a substantial number of records.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
unknown_counts = df.select_dtypes(include="object").apply(lambda s: s.eq("unknown").sum()).sort_values(ascending=False)
display(missing[missing.gt(0)].rename("missing_count").to_frame())
display(unknown_counts[unknown_counts.gt(0)].rename("unknown_count").to_frame())
print("Duplicate rows:", df.duplicated().sum())
print("Target classes:", df["y"].unique())

## 4. Exploratory data analysis

The plots below examine customer demographics, campaign history, and macroeconomic context. Because the positive class is uncommon, the conversion-rate plots are often more informative than raw counts.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for col, ax in zip(["age", "campaign", "pdays", "previous"], axes.flat):
    sns.histplot(data=df, x=col, hue="y", bins=30, stat="density",
                 common_norm=False, element="step", ax=ax)
    ax.set_title(f"Distribution of {col} by subscription")
plt.tight_layout()
plt.show()

In [ ]:
demographic_cols = ["job", "marital", "education"]
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for col, ax in zip(demographic_cols, axes):
    rates = df.groupby(col, observed=True)["y"].apply(lambda s: (s == "yes").mean()).sort_values()
    sns.barplot(x=rates.values, y=rates.index, ax=ax)
    ax.set_title(f"Subscription rate by {col}")
    ax.set_xlabel("Share subscribing")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
campaign_cols = ["contact", "month", "poutcome", "day_of_week"]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for col, ax in zip(campaign_cols, axes.flat):
    rates = df.groupby(col, observed=True)["y"].apply(lambda s: (s == "yes").mean()).sort_values()
    sns.barplot(x=rates.values, y=rates.index, ax=ax)
    ax.set_title(f"Subscription rate by {col}")
    ax.set_xlabel("Share subscribing")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
macro_cols = ["emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for col, ax in zip(macro_cols, axes.flat):
    sns.boxplot(data=df, x="y", y=col, ax=ax)
    ax.set_title(f"{col} by target")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
numeric_for_corr = df.select_dtypes(include=np.number)
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_for_corr.corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation heatmap — numeric predictors")
plt.tight_layout()
plt.show()

### Initial observations

- The target is imbalanced; the `no` class dominates.
- `poutcome` and prior campaign history may be useful signals, but should be checked for stability and operational availability.
- Macroeconomic variables move together and may be correlated; regularized models can help manage redundant signals.
- Campaign contact counts should be interpreted as historical association, not proof that additional calls cause lower conversion. Contact strategy may be confounded by customer selection and campaign policy.
- Demographic conversion-rate differences are descriptive and should not be used as a substitute for fair, compliant marketing policy.

## 5. Leakage review and modeling features

`duration` is excluded from all primary models because it is observed only after the call. Including it would leak post-decision information into a model intended to select whom to call.

The remaining fields are available from the dataset at the campaign-contact stage. `pdays`, `previous`, and `poutcome` describe previous campaign history, not the current call outcome.

In [ ]:
TARGET = "y"
LEAKAGE_FEATURE = "duration"
X = df.drop(columns=[TARGET, LEAKAGE_FEATURE])
y = (df[TARGET] == "yes").astype(int)

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
print("Categorical:", categorical_features)
print("Numeric:", numeric_features)
print("Positive class rate:", round(y.mean(), 4))

## 6. Train/test split and preprocessing pipeline

A stratified 80/20 split preserves the class ratio. All imputing, encoding, and scaling are fitted inside a scikit-learn pipeline, so cross-validation never learns preprocessing parameters from its validation fold.

Numeric variables are median-imputed and standardized. Categorical variables are imputed with the most frequent value and one-hot encoded; unknown categories remain explicit levels.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))

## 7. Baseline classifier

The dummy classifier predicts the majority class. Its accuracy is expected to be close to 88.7%, but it has no ability to identify subscribers. This illustrates why accuracy is not the primary metric.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
baseline = {
    "accuracy": accuracy_score(y_test, dummy_pred),
    "precision": precision_score(y_test, dummy_pred, zero_division=0),
    "recall": recall_score(y_test, dummy_pred, zero_division=0),
    "f1": f1_score(y_test, dummy_pred, zero_division=0)
}
pd.Series(baseline, name="Dummy majority baseline").to_frame()

## 8. Baseline comparison — four required classifiers

The SVC baseline uses a linear kernel to provide a practical SVM benchmark on the full training set. The four pipelines share identical preprocessing. Fit time is measured on the training split.

In [ ]:
baseline_models = {
    "KNN": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "SVM (linear)": SVC(kernel="linear", class_weight="balanced", random_state=RANDOM_STATE)
}

baseline_rows = []
baseline_fitted = {}
for name, estimator in baseline_models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", estimator)])
    start = time.perf_counter()
    pipe.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start
    pred = pipe.predict(X_test)
    score = pipe.decision_function(X_test) if hasattr(pipe, "decision_function") else pipe.predict_proba(X_test)[:, 1]
    baseline_rows.append({
        "Model": name, "Fit time (s)": fit_seconds,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, score),
        "PR-AUC": average_precision_score(y_test, score)
    })
    baseline_fitted[name] = pipe

baseline_results = pd.DataFrame(baseline_rows).set_index("Model").sort_values("ROC-AUC", ascending=False)
display(baseline_results.round(4))

## 9. Hyperparameter tuning with 5-fold stratified cross-validation

All searches optimize ROC-AUC. The grids are intentionally bounded to keep the notebook practical on a standard laptop. For SVM, a stratified 6,000-row sample from the training split is used for the grid search because kernel SVC can be computationally expensive on the full 32,000-row training set. The selected SVM is then refit on the full training split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search_spaces = {
    "KNN": (
        KNeighborsClassifier(),
        {"model__n_neighbors": [7, 15, 25],
         "model__weights": ["uniform", "distance"],
         "model__metric": ["minkowski"]}
    ),
    "Logistic Regression": (
        LogisticRegression(max_iter=2500, class_weight="balanced", random_state=RANDOM_STATE),
        {"model__C": [0.1, 1.0, 10.0],
         "model__solver": ["lbfgs"]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
        {"model__max_depth": [4, 8, 14, None],
         "model__min_samples_split": [10, 30],
         "model__min_samples_leaf": [5, 15],
         "model__criterion": ["gini", "entropy"]}
    )
}

searches = {}
tuning_rows = []
for name, (estimator, params) in search_spaces.items():
    pipe = Pipeline([("prep", preprocessor), ("model", estimator)])
    search = GridSearchCV(pipe, params, scoring="roc_auc", cv=cv, n_jobs=-1, refit=True, verbose=0)
    start = time.perf_counter()
    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    searches[name] = search
    tuning_rows.append({"Model": name, "Best CV ROC-AUC": search.best_score_,
                        "Search time (s)": elapsed, "Best parameters": search.best_params_})

# SVC grid on a stratified training subset; refit selected configuration on full training data.
X_svm, _, y_svm, _ = train_test_split(
    X_train, y_train, train_size=min(6000, len(X_train)),
    random_state=RANDOM_STATE, stratify=y_train
)
svm_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(class_weight="balanced", random_state=RANDOM_STATE))
])
svm_params = {
    "model__kernel": ["linear", "rbf"],
    "model__C": [0.5, 2.0],
    "model__gamma": ["scale"]
}
svm_search = GridSearchCV(svm_pipe, svm_params, scoring="roc_auc", cv=cv,
                          n_jobs=-1, refit=False, verbose=0)
start = time.perf_counter()
svm_search.fit(X_svm, y_svm)
svm_elapsed = time.perf_counter() - start
best_svm = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(**{k.replace("model__", ""): v for k, v in svm_search.best_params_.items()},
                  class_weight="balanced", random_state=RANDOM_STATE))
])
best_svm.fit(X_train, y_train)
searches["SVM"] = svm_search
tuning_rows.append({"Model": "SVM", "Best CV ROC-AUC (6k sample)": svm_search.best_score_,
                    "Search time (s)": svm_elapsed, "Best parameters": svm_search.best_params_})

tuning_summary = pd.DataFrame(tuning_rows).set_index("Model")
display(tuning_summary)

## 10. Final holdout evaluation

The test set is used once for final model comparison after model selection. Threshold-based metrics use the default 0.5 threshold (or the estimator's default decision boundary). A real campaign should choose a threshold based on contact capacity and the cost/value trade-off, using validation data rather than tuning on the test set.

In [ ]:
final_models = {name: search.best_estimator_ for name, search in searches.items() if name != "SVM"}
final_models["SVM"] = best_svm

final_rows = []
test_scores = {}
test_predictions = {}
for name, model in final_models.items():
    pred = model.predict(X_test)
    score = model.decision_function(X_test) if hasattr(model, "decision_function") else model.predict_proba(X_test)[:, 1]
    test_predictions[name] = pred
    test_scores[name] = score
    final_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, score),
        "PR-AUC": average_precision_score(y_test, score)
    })

final_results = pd.DataFrame(final_rows).set_index("Model").sort_values("ROC-AUC", ascending=False)
display(final_results.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for name, scores in test_scores.items():
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test, scores)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, scores):.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random")
ax.set(title="ROC curves — holdout test set", xlabel="False positive rate", ylabel="True positive rate")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, pred) in zip(axes.flat, test_predictions.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["No", "Yes"],
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 11. Model interpretation

### Logistic Regression — odds ratios

Coefficients are interpreted on the standardized numeric / one-hot encoded feature scale. Exponentiated coefficients represent multiplicative changes in odds for a one-unit increase in the transformed feature, holding other features fixed. These are associations, not causal effects.

In [ ]:
lr = final_models["Logistic Regression"]
feature_names = lr.named_steps["prep"].get_feature_names_out()
coefs = lr.named_steps["model"].coef_.ravel()
lr_interpretation = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefs,
    "odds_ratio": np.exp(np.clip(coefs, -20, 20))
}).sort_values("coefficient", ascending=False)

print("Largest positive associations:")
display(lr_interpretation.head(15).round(3))
print("Largest negative associations:")
display(lr_interpretation.tail(15).sort_values("coefficient").round(3))

### Decision Tree — feature importance and representative rules

In [ ]:
tree = final_models["Decision Tree"]
tree_names = tree.named_steps["prep"].get_feature_names_out()
tree_importance = pd.Series(tree.named_steps["model"].feature_importances_, index=tree_names)
tree_importance = tree_importance.sort_values(ascending=False)
display(tree_importance.head(20).rename("importance").to_frame())

plt.figure(figsize=(22, 10))
plot_tree(tree.named_steps["model"], feature_names=tree_names, class_names=["No", "Yes"],
          filled=True, rounded=True, max_depth=3, fontsize=8)
plt.title("Decision Tree — top three levels")
plt.tight_layout()
plt.show()

## 12. Duration leakage sensitivity check

This optional diagnostic compares logistic regression with and without `duration`. It is deliberately labelled **post-call diagnostic**, not a deployable model. A large performance increase when duration is included is expected and demonstrates why it must not be used for pre-call targeting.

In [ ]:
X_with_duration = df.drop(columns=[TARGET])
Xtr_d, Xte_d, ytr_d, yte_d = train_test_split(
    X_with_duration, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
cat_d = X_with_duration.select_dtypes(include=["object"]).columns.tolist()
num_d = X_with_duration.select_dtypes(include=np.number).columns.tolist()
prep_d = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_d),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_d)
])
duration_model = Pipeline([("prep", prep_d),
                           ("model", LogisticRegression(max_iter=2500, class_weight="balanced"))])
duration_model.fit(Xtr_d, ytr_d)
duration_score = duration_model.decision_function(Xte_d)
duration_pred = duration_model.predict(Xte_d)

leakage_comparison = pd.DataFrame([
    {"Feature set": "Pre-call (duration excluded)",
     "ROC-AUC": roc_auc_score(y_test, test_scores["Logistic Regression"]),
     "PR-AUC": average_precision_score(y_test, test_scores["Logistic Regression"])},
    {"Feature set": "Post-call diagnostic (duration included)",
     "ROC-AUC": roc_auc_score(yte_d, duration_score),
     "PR-AUC": average_precision_score(yte_d, duration_score)}
]).set_index("Feature set")
display(leakage_comparison.round(4))

## 13. Business findings and recommendations

Use the actual model results and EDA above to finalize campaign policy. The following recommendations are decision-oriented and should be validated with a controlled pilot:

1. **Prioritize by ranked propensity, not a fixed demographic stereotype.** Use the selected model to rank prospects and test the top-scoring segment against the current campaign strategy.
2. **Respect capacity and economics.** Choose a contact threshold or top-*k* list based on agent capacity, average call cost, and expected value of a term-deposit subscription. Report precision and expected conversions per 100 calls.
3. **Use prior campaign outcomes carefully.** Customers with a previous successful outcome (`poutcome`) may be high-priority candidates, subject to recency and consent rules.
4. **Limit repeated attempts.** Investigate diminishing returns as `campaign` increases. Establish a contact cap and suppression rules rather than repeatedly calling low-propensity leads.
5. **Time campaigns with macro context.** Review associations between conversion and `euribor3m`, `emp.var.rate`, and related indicators. These are predictive associations in this historical dataset, not causal timing prescriptions.
6. **Pilot and monitor.** Run an A/B test, track conversion lift, cost per subscription, complaints/opt-outs, and performance drift. Recalibrate the threshold as campaign economics change.

### Limitations

The dataset is historical and reflects a particular bank, period, and campaign process. Correlations can reflect selection effects and campaign policy. The model does not establish causal effects, and historical patterns may not generalize to a new market or time period. Fairness, privacy, consent, and contact-frequency requirements must be reviewed before deployment.

## 14. Conclusion

The recommended model is the one with the strongest holdout ranking performance **and** acceptable precision/recall at a threshold that matches business capacity. Do not select a model solely by accuracy. The final recommendation should cite the actual `final_results` table and explain the precision–recall trade-off. The `duration`-excluded pipeline is the appropriate candidate for a pre-call lead-prioritization workflow.

## References

- UCI Machine Learning Repository, [Bank Marketing Data Set](https://archive.ics.uci.edu/dataset/222/bank+marketing).
- Moro, S., Laureano, R., & Cortez, P. (2011). *Using Data Mining for Bank Direct Marketing: An Application of the CRISP-DM Methodology*. Included project reference: `CRISP-DM-BANK.pdf`.
- scikit-learn documentation: [Pipeline](https://scikit-learn.org/stable/modules/compose.html#pipeline), [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), [classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html).